In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("KafkaSparkStreaming")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark is ready")

:: loading settings :: url = jar:file:/Users/Bootcamp/kafka-spark-streaming-lab/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/Bootcamp/.ivy2/cache
The jars for the packages stored in: /Users/Bootcamp/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a665f8c7-81c5-479a-aa3e-d99a34a8bdd8;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.0 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
downloading https://repo1.maven.org/maven2/org/apache/sp

Spark is ready


In [2]:
df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "orders")
    .option("startingOffsets", "earliest")
    .load()
)

print("Streaming:", df.isStreaming)

Streaming: True


In [3]:
orders_df = df.selectExpr(
    "CAST(value AS STRING) AS value"
)

orders_df.printSchema()

root
 |-- value: string (nullable = true)



In [4]:
query = (
    orders_df.writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", "false")
    .start()
)

print("Query running:", query.isActive)

Query running: True


-------------------------------------------
Batch: 0
-------------------------------------------
+-------------------------------------+
|value                                |
+-------------------------------------+
|{"customer": "John", "amount": 500}  |
|{"customer": "Sara", "amount": 1200} |
|{"customer": "Mike", "amount": 300}  |
|{"customer": "Alice", "amount": 1500}|
+-------------------------------------+

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------------------------------+
|value                              |
+-----------------------------------+
|{"customer": "John", "amount": 500}|
+-----------------------------------+

-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------+
|value                               |
+------------------------------------+
|{"customer": "Sara", "amount": 1200}|
+------------------------------------+

-

In [5]:
query.stop()

print("Query running:", query.isActive)

Query running: False


In [6]:
from pyspark.sql.functions import col, from_json, when
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

order_schema = StructType([
    StructField("customer", StringType(), True),
    StructField("amount", IntegerType(), True)
])

parsed_df = (
    orders_df
    .select(from_json(col("value"), order_schema).alias("order"))
    .select("order.*")
)

high_value_df = parsed_df.withColumn(
    "high_value",
    when(col("amount") > 1000, "YES").otherwise("NO")
)

high_value_df.printSchema()

root
 |-- customer: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- high_value: string (nullable = false)



In [7]:
high_value_query = (
    high_value_df.writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", "false")
    .start()
)

print("High-value query running:", high_value_query.isActive)


High-value query running: True


-------------------------------------------
Batch: 0
-------------------------------------------
+--------+------+----------+
|customer|amount|high_value|
+--------+------+----------+
|John    |500   |NO        |
|Sara    |1200  |YES       |
|Mike    |300   |NO        |
|Alice   |1500  |YES       |
|John    |500   |NO        |
|Sara    |1200  |YES       |
|Mike    |300   |NO        |
|Alice   |1500  |YES       |
+--------+------+----------+



In [8]:
high_value_query.stop()

print("High-value query running:", high_value_query.isActive)

High-value query running: False
